# Geosteering AI — Sprint A1: Validação JAX GPU

**Template:** `validate_jax_gpu_v240.ipynb` | **Sprint:** A1 (`A-jax-gpu-validate`)

Valida o simulador JAX GPU em hardware real (T4 e A100), mede throughput nos
cenários canônicos A–H e publica baseline empírico para `docs/PERFORMANCE_BASELINE.md`.

**Pré-requisito:** Runtime GPU T4 ou superior (Runtime → Change runtime type → GPU).

| Lição Numba | Onde aplicada |
|:--|:--|
| L5 — Configurações JAX antes de qualquer import | Célula 1 (primeira do notebook) |
| L8 — Cache JIT em SSD rápido (`/content/jax_cache`) | Célula 1 + Célula 3 |
| L4 — Warmup com dados reais (anisotrópico + dip ≠ 0°) | Seção 3 |

**Critérios de aprovação:**
- Throughput JAX GPU ≥ 1.5× Numba CPU nos cenários A, B, E (mínimo)
- `pytest tests/test_simulation_jax_*.py -m gpu` → 100% PASS
- Sem OOM para n_models=50, n_pos=600 (Cenário E)

In [ ]:
# === L5: Configurações JAX OBRIGATÓRIAS antes de qualquer import ===
# Equivalente ao 'NUMBA_NUM_THREADS no PAI' (L5 do mapeamento Numba→JAX):
# jax_enable_x64 e jax_compilation_cache_dir devem ser definidos ANTES
# de 'import jax' — após o import, jax_enable_x64 fica imutável.
import os

os.environ["JAX_ENABLE_X64"] = "1"                              # float64 — paridade Fortran <1e-12 requer float64
os.environ["JAX_PLATFORMS"] = "" # auto-detect: inicializa CPU+GPU (pure_callback do kernel.py requer CPU)
os.environ["JAX_COMPILATION_CACHE_DIR"] = "/content/jax_cache"  # L8: cache XLA em SSD Colab

print("JAX env vars configurados (antes de qualquer import):")
print(f"  JAX_ENABLE_X64             = {os.environ['JAX_ENABLE_X64']}")
print(f"  JAX_PLATFORMS              = {os.environ['JAX_PLATFORMS']}")
print(f"  JAX_COMPILATION_CACHE_DIR  = {os.environ['JAX_COMPILATION_CACHE_DIR']}")

## § 1 — Configurações da Sprint

In [ ]:
GIT_TAG           = "v2.40"                                                   # tag do repositório
OUTPUT_DRIVE_DIR  = "/content/drive/MyDrive/Geosteering_AI/sprint_a1/"        # destino dos resultados
PARIDADE_TOL      = 1e-12   # gate de paridade rigoroso (vs 1e-10 do template v2.40)
N_RUNS            = 5       # repetições por cenário × modo para estatística de throughput
N_MODELS          = 50      # modelos geológicos por run (5 camadas, semente 42)
SEED              = 42      # reprodutibilidade

# Diretório de cache JAX (L8): deve existir antes de import jax
os.makedirs("/content/jax_cache", exist_ok=True)

print(f"GIT_TAG={GIT_TAG} | N_MODELS={N_MODELS} | N_RUNS={N_RUNS} | tol={PARIDADE_TOL:.0e}")

## § 2 — Setup do Ambiente

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Clonar repositório e instalar dependências
!git clone --depth 1 --branch {GIT_TAG} https://github.com/daniel-guitarplayer-8/geosteering-ai.git
%cd geosteering-ai
!pip install -q -e ".[all]"
!pip install -q pytest-json-report

# Imports — apenas APÓS as env vars da Célula 1
import json
import datetime
import statistics
import subprocess
import time

import jax
import jax.numpy as jnp
import numpy as np

# Validar GPU
devices = jax.devices()
gpu_devs = [d for d in devices if d.platform == "gpu"]
assert gpu_devs, f"GPU não detectada! Devices: {devices}. Verificar Runtime → GPU."
print(f"Dispositivo GPU: {gpu_devs[0]}")

# Validar float64
assert jax.config.jax_enable_x64, (
    "ERRO CRÍTICO: float64 não habilitado! "
    "JAX_ENABLE_X64 deve ser setado ANTES de 'import jax' (Célula 1)."
)
print(f"float64 habilitado: {jax.config.jax_enable_x64} ✓")

# Hardware info
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

# Pin commit hash (imutável — tag pode ser movida, hash não)
try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd="/content/geosteering-ai"
    ).decode().strip()
except (subprocess.CalledProcessError, FileNotFoundError):
    GIT_COMMIT = f"unknown-{GIT_TAG}"
print(f"Commit: {GIT_COMMIT[:12]}")

## § 3 — Warmup L4 (Dados Reais, Antes de Qualquer Medição)

**Por que L4 é crítico:** o simulador Numba JIT aprendeu (Sprint v2.28) que warmup
com dados isotrópicos + dip=0° deixa 5 funções JIT não compiladas (elas só são
acionadas com dados anisotrópicos ou dip ≠ 0°). A mesma lição se aplica ao JAX:
o compilador XLA gera traces distintos por shape — aqui aquecemos todos os shapes
de n_pos usados nos cenários A–H: {1, 100, 600}.

O warmup cobre ambos os modos (`jax_vmap_real=False` e `True`) pois usam caminhos
JIT distintos. O tempo de compile é esperado (1ª chamada) e não conta para benchmark.

In [ ]:
from geosteering_ai.simulation._jax.multi_forward import simulate_multi_jax
from geosteering_ai.simulation.config import SimulationConfig

# Modelo anisotrópico com dip ≠ 0° (cobre paths JIT que isotropico+dip=0 não acionam — L4)
_RHO_H_WARM = np.array([2.0, 15.0, 120.0, 8.0, 40.0], dtype=np.float64)
_RHO_V_WARM = _RHO_H_WARM * np.array([1.8, 2.5, 3.0, 1.5, 2.0], dtype=np.float64)
_ESP_WARM   = np.array([6.0, 9.0, 7.0], dtype=np.float64)   # 5 camadas → 3 espessuras internas

_WARMUP_FREQS = [2000.0, 20000.0]      # 2 frequências
_WARMUP_TRS   = [0.5, 1.0, 1.5]       # 3 TR spacings
_WARMUP_DIPS  = [0.0, 30.0]           # dip=30° aciona path anisotrópico

_N_POS_SHAPES = [
    (1,   "Cenários A, D"),
    (100, "Cenários B, C, F, G, H"),
    (600, "Cenário E"),
]

print("=== Warmup L4 — modo vmap (jax_vmap_real=False) ===")
_cfg_loop = SimulationConfig(jax_vmap_real=False)
for _n_pos, _desc in _N_POS_SHAPES:
    _pos = np.linspace(-_n_pos / 2.0, _n_pos / 2.0, _n_pos)
    t0 = time.perf_counter()
    simulate_multi_jax(_RHO_H_WARM, _RHO_V_WARM, _ESP_WARM, _pos,
                       frequencies_hz=_WARMUP_FREQS, tr_spacings_m=_WARMUP_TRS,
                       dip_degs=_WARMUP_DIPS, cfg=_cfg_loop)
    # 2ª chamada confirma hot-cache (sem recompile)
    simulate_multi_jax(_RHO_H_WARM, _RHO_V_WARM, _ESP_WARM, _pos,
                       frequencies_hz=_WARMUP_FREQS, tr_spacings_m=_WARMUP_TRS,
                       dip_degs=_WARMUP_DIPS, cfg=_cfg_loop)
    print(f"  n_pos={_n_pos:4d} ({_desc}): {time.perf_counter()-t0:.1f}s (inclui compile XLA)")

print("\n=== Warmup L4 — modo vmap_real (jax_vmap_real=True) ===")
_cfg_vr = SimulationConfig(jax_vmap_real=True)
for _n_pos, _desc in _N_POS_SHAPES:
    _pos = np.linspace(-_n_pos / 2.0, _n_pos / 2.0, _n_pos)
    t0 = time.perf_counter()
    simulate_multi_jax(_RHO_H_WARM, _RHO_V_WARM, _ESP_WARM, _pos,
                       frequencies_hz=_WARMUP_FREQS, tr_spacings_m=_WARMUP_TRS,
                       dip_degs=_WARMUP_DIPS, cfg=_cfg_vr)
    simulate_multi_jax(_RHO_H_WARM, _RHO_V_WARM, _ESP_WARM, _pos,
                       frequencies_hz=_WARMUP_FREQS, tr_spacings_m=_WARMUP_TRS,
                       dip_degs=_WARMUP_DIPS, cfg=_cfg_vr)
    print(f"  n_pos={_n_pos:4d} ({_desc}): {time.perf_counter()-t0:.1f}s")

print("\nWarmup concluído. Benchmark a seguir mede apenas hot-cache performance.")

## § 4 — Paridade Fortran < 1e-12 (109 Testes JAX)

Os 109 testes `@pytest.mark.gpu` validam paridade JAX GPU vs Fortran < 1e-12.
Com `JAX_PLATFORMS="" (auto-detect)` (Célula 1), o conftest detecta GPU e executa os testes
que seriam SKIPPED em CPU.

In [ ]:
# Executa 109 testes JAX em GPU com relatório JSON
!python -m pytest tests/test_simulation_jax_*.py \
    -m gpu \
    -v \
    --tb=short \
    --json-report \
    --json-report-file=/tmp/jax_gpu_pytest.json \
    2>&1 | tail -40

# Verificação de gate
with open("/tmp/jax_gpu_pytest.json") as _f:
    _pytest_report = json.load(_f)
_summary = _pytest_report["summary"]
print(f"\nResumo pytest: {_summary}")
assert _summary.get("failed", 0) == 0, (
    f"GATE PARIDADE FALHOU: {_summary['failed']} testes falharam. "
    "Investigar antes de prosseguir com benchmark."
)
print(f"✓ Paridade: {_summary.get('passed', 0)} PASS / 0 FAIL")

In [ ]:
# Verificação rápida de shape, dtype e ausência de NaN/Inf
_cfg_check = SimulationConfig(jax_vmap_real=False)
_rho_h = np.array([1.0, 10.0, 100.0, 10.0, 1.0], dtype=np.float64)
_rho_v = _rho_h * 2.0
_esp   = np.array([5.0, 10.0, 5.0], dtype=np.float64)
_pos   = np.linspace(-10.0, 10.0, 50)

_res = simulate_multi_jax(
    _rho_h, _rho_v, _esp, _pos,
    frequencies_hz=[20000.0], tr_spacings_m=[1.0], dip_degs=[0.0],
    cfg=_cfg_check,
)

assert _res.H_tensor.dtype  == np.complex128,          f"dtype errado: {_res.H_tensor.dtype}"
assert _res.H_tensor.shape  == (1, 1, 50, 1, 9),       f"shape errado: {_res.H_tensor.shape}"
assert not np.any(np.isnan(_res.H_tensor.real)),        "NaN em H_tensor.real!"
assert not np.any(np.isnan(_res.H_tensor.imag)),        "NaN em H_tensor.imag!"
assert not np.any(np.isinf(_res.H_tensor)),             "Inf em H_tensor!"

print(f"Shape:  {_res.H_tensor.shape}  ✓")
print(f"dtype:  {_res.H_tensor.dtype}  ✓")
print(f"NaN:    ausente  ✓")
print(f"Inf:    ausente  ✓")
print(f"|Hzz| médio: {np.abs(_res.H_tensor[0,0,:,0,8]).mean():.6e}  (componente axial ZZ)")

## § 5 — Benchmark Cenários A–H

Mede throughput em **mod/h** para dois modos JAX GPU:
- **vmap** (`jax_vmap_real=False`): Python loop com dedup por hordist (default)
- **vmap_real** (`jax_vmap_real=True`): `jax.vmap` sobre (iTR × iAng), 1 único JIT

Metodologia: N_MODELS modelos geológicos de 5 camadas (semente 42), gerados por
`_build_models()`. Cada run cronometrado com `time.perf_counter()`. A sincronização
JAX é garantida pela conversão `np.asarray()` interna ao `simulate_multi_jax`.

**Nota Cenário H:** validador JAX aceita `dip_degs ≤ 89.0°`; benchmark Numba usa
90.0° (fix v2.36 D2). JAX usa 87.5° para manter 8 dips sem ultrapassar o limite.
Diferença insignificante para throughput — apenas anotar nos resultados.

In [ ]:
# --- Cenários canônicos (espelha geosteering_ai/cli/benchmark.py:113-147) ---
# dips_jax: versão segura para validador JAX (max 89.0°)
# numba_baseline_mod_h: referência histórica Numba CPU 8C/16T (None = não medido)
SCENARIOS_JAX = {
    "A": {
        "n_pos": 1,
        "freqs": (20000.0,),
        "trs":   (1.0,),
        "dips_jax": (0.0,),
        "numba_baseline_mod_h": 1_180_000,
    },
    "B": {
        "n_pos": 100,
        "freqs": (20000.0,),
        "trs":   (1.0,),
        "dips_jax": (0.0,),
        "numba_baseline_mod_h": 320_000,
    },
    "C": {
        "n_pos": 100,
        "freqs": (2000.0, 20000.0, 100000.0, 400000.0),
        "trs":   (1.0,),
        "dips_jax": (0.0,),
        "numba_baseline_mod_h": 119_000,
    },
    "D": {
        "n_pos": 1,
        "freqs": (20000.0,),
        "trs":   (0.5, 1.0, 1.5, 2.0),
        "dips_jax": (0.0,),
        "numba_baseline_mod_h": None,   # baseline Numba medido em ms/run: ~13.4ms
    },
    "E": {
        "n_pos": 600,
        "freqs": (20000.0,),
        "trs":   (1.0,),
        "dips_jax": (0.0,),
        "numba_baseline_mod_h": 122_000,
    },
    "F": {
        "n_pos": 100,
        "freqs": (2000.0, 20000.0, 100000.0, 400000.0),
        "trs":   (0.5, 1.0, 1.5, 2.0),
        "dips_jax": (0.0,),
        "numba_baseline_mod_h": None,
    },
    "G": {
        "n_pos": 100,
        "freqs": (2000.0, 20000.0, 100000.0, 400000.0),
        "trs":   (0.5, 1.0, 1.5, 2.0),
        "dips_jax": (0.0, 15.0, 30.0, 45.0),
        "numba_baseline_mod_h": 45_000,
    },
    "H": {
        "n_pos": 100,
        "freqs": (1e3, 2e3, 5e3, 1e4, 2e4, 5e4, 1e5, 2e5),
        "trs":   (0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5),
        # 87.5° no lugar de 90.0° do Numba (limite JAX: dip_degs <= 89.0°)
        "dips_jax": (0.0, 12.5, 25.0, 37.5, 50.0, 62.5, 75.0, 87.5),
        "numba_baseline_mod_h": None,
    },
}

# Número de modelos por cenário: H tem 512 combos (8×8×8) → reduzido para controlar tempo
N_MODELS_PER_SCENARIO = {
    scen: (min(N_MODELS, 10) if scen == "H" else N_MODELS)
    for scen in SCENARIOS_JAX
}


def _build_models(n_models: int, seed: int = 42) -> list:
    """Gera n_models modelos geológicos determinísticos (5 camadas, mesmo formato do CLI benchmark)."""
    rng = np.random.default_rng(seed)
    models = []
    for _ in range(n_models):
        rho_h = rng.uniform(1.0, 100.0, size=5).astype(np.float64)
        rho_v = rho_h * rng.uniform(1.0, 3.0, size=5).astype(np.float64)
        esp   = rng.uniform(2.0, 10.0, size=3).astype(np.float64)   # n=5 → 3 espessuras internas
        models.append({"rho_h": rho_h, "rho_v": rho_v, "esp": esp})
    return models


def run_benchmark_scenario(
    scen_name: str,
    n_models: int,
    n_runs: int,
    jax_vmap_real: bool,
    seed: int = 42,
) -> dict:
    """Executa benchmark de um cenário em n_runs repetições.

    Inclui warmup por-cenário (1 modelo, não cronometrado) para garantir que
    o JIT XLA específico a este (nf, nTR, nAngles, n_pos) esteja compilado
    antes das medições. Os n_runs timed runs medem hot-cache performance.

    A sincronização JAX é garantida por np.asarray() interno ao simulate_multi_jax
    — não é necessário .block_until_ready() explícito.

    Returns dict com throughputs_mod_h, median, min, max e latência ms/modelo.
    """
    scen = SCENARIOS_JAX[scen_name]
    n_pos = scen["n_pos"]
    positions_z = np.linspace(-n_pos / 2.0, n_pos / 2.0, n_pos)
    models = _build_models(n_models, seed=seed)
    cfg = SimulationConfig(jax_vmap_real=jax_vmap_real)

    kwargs = dict(
        frequencies_hz=list(scen["freqs"]),
        tr_spacings_m=list(scen["trs"]),
        dip_degs=list(scen["dips_jax"]),
        cfg=cfg,
    )

    # Warmup por-cenário: compila JIT para este (nf, nTR, nAngles, n_pos) específico
    simulate_multi_jax(models[0]["rho_h"], models[0]["rho_v"], models[0]["esp"],
                       positions_z, **kwargs)

    throughputs = []
    for run_i in range(n_runs):
        t0 = time.perf_counter()
        for m in models:
            simulate_multi_jax(m["rho_h"], m["rho_v"], m["esp"], positions_z, **kwargs)
        elapsed = time.perf_counter() - t0
        th = n_models / elapsed * 3600.0
        throughputs.append(th)
        print(f"    Run {run_i+1}/{n_runs}: {th:>12,.0f} mod/h  ({elapsed:.3f}s)")

    med = statistics.median(throughputs)
    return {
        "scenario": scen_name,
        "vmap_real": jax_vmap_real,
        "n_models": n_models,
        "n_pos": n_pos,
        "nf": len(scen["freqs"]),
        "nTR": len(scen["trs"]),
        "nAng": len(scen["dips_jax"]),
        "n_runs": n_runs,
        "throughputs_mod_h": throughputs,
        "median_mod_h": med,
        "min_mod_h": min(throughputs),
        "max_mod_h": max(throughputs),
        "latency_ms_per_model": statistics.median(
            [n_models / th * 3_600_000 for th in throughputs]
        ),
        "numba_baseline_mod_h": scen["numba_baseline_mod_h"],
    }


print("Funções de benchmark definidas.")
print(f"Cenários: {list(SCENARIOS_JAX.keys())}")
print(f"N_MODELS por cenário: {N_MODELS_PER_SCENARIO}")

In [ ]:
# === Benchmark — Modo vmap (jax_vmap_real=False) ===
# Python loop com dedup de cache por hordist = L·|sin(θ)|
# Para dip=0° (cenários A–F), dedup colapsa todas as TRs em 1 grupo.

results_vmap = {}
print("=" * 65)
print("BENCHMARK — modo vmap (jax_vmap_real=False)")
print("=" * 65)

for scen_name in SCENARIOS_JAX:
    n_m = N_MODELS_PER_SCENARIO[scen_name]
    scen = SCENARIOS_JAX[scen_name]
    n_combos = len(scen["trs"]) * len(scen["dips_jax"]) * len(scen["freqs"])
    print(f"\nCenário {scen_name} — n_pos={scen['n_pos']}, {n_combos} combos (nTR×nAng×nf), n_models={n_m}")
    results_vmap[scen_name] = run_benchmark_scenario(
        scen_name, n_models=n_m, n_runs=N_RUNS, jax_vmap_real=False, seed=SEED
    )
    r = results_vmap[scen_name]
    baseline = r["numba_baseline_mod_h"]
    ratio_str = f"  ({r['median_mod_h']/baseline:.2f}× Numba)" if baseline else ""
    print(f"  → Mediana: {r['median_mod_h']:>12,.0f} mod/h{ratio_str}")

print("\n✓ Benchmark vmap concluído.")

In [ ]:
# === Benchmark — Modo vmap_real (jax_vmap_real=True) ===
# jax.vmap sobre grid achatado (nTR × nAngles), 1 único JIT por (n, n_pos).
# Esperado ser mais eficiente em GPU para cenários multi-TR + multi-dip (F, G, H).
# Para cenários com 1 TR × 1 dip (A, B, C, E), overhead de vmap é mínimo.

results_vmap_real = {}
print("=" * 65)
print("BENCHMARK — modo vmap_real (jax_vmap_real=True)")
print("=" * 65)

for scen_name in SCENARIOS_JAX:
    n_m = N_MODELS_PER_SCENARIO[scen_name]
    scen = SCENARIOS_JAX[scen_name]
    n_combos = len(scen["trs"]) * len(scen["dips_jax"]) * len(scen["freqs"])
    print(f"\nCenário {scen_name} — n_pos={scen['n_pos']}, {n_combos} combos (nTR×nAng×nf), n_models={n_m}")
    results_vmap_real[scen_name] = run_benchmark_scenario(
        scen_name, n_models=n_m, n_runs=N_RUNS, jax_vmap_real=True, seed=SEED
    )
    r = results_vmap_real[scen_name]
    baseline = r["numba_baseline_mod_h"]
    ratio_str = f"  ({r['median_mod_h']/baseline:.2f}× Numba)" if baseline else ""
    print(f"  → Mediana: {r['median_mod_h']:>12,.0f} mod/h{ratio_str}")

print("\n✓ Benchmark vmap_real concluído.")

In [ ]:
# === Tabela Comparativa: vmap vs vmap_real vs Numba ===
print("=" * 100)
print(f"{'Cen':>3} | {'n_pos':>5} | {'nf×TR×Ang':>9} | {'Numba (baseline)':>18} | {'JAX vmap (med)':>16} | {'JAX vmap_real (med)':>19} | {'Razão vmap':>10} | {'Razão vmap_real':>15}")
print("-" * 100)

GATE_PASS_SCENARIOS = ["A", "B", "E"]  # cenários com critério de aprovação ≥1.5×
gate_results = {}

for scen_name in SCENARIOS_JAX:
    scen   = SCENARIOS_JAX[scen_name]
    rv     = results_vmap[scen_name]
    rvr    = results_vmap_real[scen_name]
    nb     = scen["numba_baseline_mod_h"]
    combos = f"{len(scen['freqs'])}×{len(scen['trs'])}×{len(scen['dips_jax'])}"

    med_v  = rv["median_mod_h"]
    med_vr = rvr["median_mod_h"]

    nb_str    = f"{nb:>14,.0f}" if nb else f"{'(medir)':>14}"
    ratio_v   = f"{med_v/nb:.2f}×"   if nb else "—"
    ratio_vr  = f"{med_vr/nb:.2f}×"  if nb else "—"

    print(f"{scen_name:>3} | {scen['n_pos']:>5} | {combos:>9} | {nb_str:>18} | {med_v:>14,.0f} m/h | {med_vr:>17,.0f} m/h | {ratio_v:>10} | {ratio_vr:>15}")

    if scen_name in GATE_PASS_SCENARIOS and nb:
        best = max(med_v, med_vr)
        gate_results[scen_name] = {"best_mod_h": best, "ratio": best / nb, "pass": best / nb >= 1.5}

print("=" * 100)

print("\n=== Gate de Aprovação (≥ 1.5× Numba em A, B, E) ===")
all_pass = True
for scen_name, g in gate_results.items():
    status = "✓ PASS" if g["pass"] else "✗ FAIL"
    print(f"  Cenário {scen_name}: {g['best_mod_h']:>12,.0f} mod/h ({g['ratio']:.2f}× Numba) → {status}")
    if not g["pass"]:
        all_pass = False

if all_pass:
    print("\n✓ GATE APROVADO: JAX GPU supera 1.5× Numba nos cenários A, B, E.")
else:
    print("\n✗ GATE FALHOU: revisar gaps G1-G8 antes de prosseguir para Sprint A2.")

## § 6 — Exportar Resultados para Drive

In [ ]:
# Detectar hardware (T4 / A100 / etc.)
try:
    _gpu_name = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]
    ).decode().strip().lower()
    RUNTIME_LABEL = "a100" if "a100" in _gpu_name else ("t4" if "t4" in _gpu_name else _gpu_name.replace(" ", "_"))
except Exception:
    RUNTIME_LABEL = "gpu_unknown"
print(f"Hardware detectado: {RUNTIME_LABEL}")

ts = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
os.makedirs(OUTPUT_DRIVE_DIR, exist_ok=True)
out_fname = f"sprint_a1_jax_benchmark_{RUNTIME_LABEL}_{ts}.json"
out_path  = os.path.join(OUTPUT_DRIVE_DIR, out_fname)

# Carregar relatório pytest
with open("/tmp/jax_gpu_pytest.json") as _f:
    _pytest_data = json.load(_f)

output = {
    "sprint": "A1 — A-jax-gpu-validate",
    "template": "validate_jax_gpu_v240",
    "git_tag": GIT_TAG,
    "git_commit": GIT_COMMIT,
    "runtime": RUNTIME_LABEL,
    "timestamp_utc": ts,
    "config": {
        "n_models_per_scenario": N_MODELS_PER_SCENARIO,
        "n_runs": N_RUNS,
        "seed": SEED,
        "paridade_tol": PARIDADE_TOL,
        "jax_enable_x64": True,
    },
    "pytest_summary": _pytest_data["summary"],
    "pytest_duration_sec": _pytest_data.get("duration", 0.0),
    "benchmark_vmap": results_vmap,
    "benchmark_vmap_real": results_vmap_real,
    "gate_results": gate_results,
    "gate_pass": all_pass,
    "h_scenario_note": "dip_degs max=87.5° (JAX) vs 90.0° (Numba benchmark v2.36 D2)",
}

with open(out_path, "w") as _f:
    json.dump(output, _f, indent=2)

print(f"✓ Resultados exportados: {out_path}")
print(f"  Tamanho: {os.path.getsize(out_path) / 1024:.1f} KB")

## § 7 — Resumo Final e Próximos Passos

In [ ]:
_pytest_sum = _pytest_data["summary"]

print("=" * 70)
print("SPRINT A1 — VALIDAÇÃO JAX GPU CONCLUÍDA")
print(f"Hardware: {RUNTIME_LABEL.upper()} | Commit: {GIT_COMMIT[:12]}")
print("=" * 70)
print(f"\nParidade Fortran <1e-12 (pytest):")
print(f"  PASS:    {_pytest_sum.get('passed', 0)}")
print(f"  FAIL:    {_pytest_sum.get('failed', 0)}")
print(f"  SKIPPED: {_pytest_sum.get('skipped', 0)}")

print("\nThroughput JAX GPU vs Numba CPU (mediana, melhor modo):")
for scen_name in SCENARIOS_JAX:
    rv  = results_vmap[scen_name]["median_mod_h"]
    rvr = results_vmap_real[scen_name]["median_mod_h"]
    nb  = SCENARIOS_JAX[scen_name]["numba_baseline_mod_h"]
    best = max(rv, rvr)
    best_mode = "vmap_real" if rvr >= rv else "vmap"
    ratio_str = f"  ({best/nb:.2f}× Numba)" if nb else ""
    print(f"  Cenário {scen_name}: {best:>12,.0f} mod/h [{best_mode}]{ratio_str}")

print(f"\nGate ≥1.5× Numba (A, B, E): {'✓ APROVADO' if all_pass else '✗ REPROVADO'}")
print(f"\nResultados salvos em: {out_path}")
print("=" * 70)

print("""
PRÓXIMOS PASSOS:
  1. Reportar tabela de resultados para nova sessão Claude
  2. Claude preenche docs/PERFORMANCE_BASELINE.md seção JAX GPU
  3. Claude fecha Sprint A1 → cria docs/sprints/v2.X.md
  4. Claude atualiza ROADMAP §0: A-jax-gpu-validate → DONE
  5. Claude inicia Sprint A2: implement simulation.simulate(backend=...)
""")